In [ ]:
import os
from google.colab import drive

# 1. Mount Google Drive securely
drive.mount('/content/drive')

# 2. Define the absolute path to your project root
PROJECT_ROOT = "/content/drive/MyDrive/KOMOREBI"

# 3. Define the exact subdirectories required for the v2 pipeline
folders = [
    "data",               # For raw Japanese audio files (.wav)
    "transcripts",        # For the Whisper ASR text outputs
    "checkpoints",        # To save model weights during fine-tuning
    "logs",               # For training metrics (loss, accuracy, AUC)
    "knowledge_graphs"    # For the Part B RAG database (Showa-era context)
]

# 4. Create the architecture programmatically
print("Initializing Project KOMOREBI directory structure...\n")
for folder in folders:
    folder_path = os.path.join(PROJECT_ROOT, folder)
    os.makedirs(folder_path, exist_ok=True) # exist_ok=True prevents errors if the folder already exists
    print(f"📁 Ready: {folder_path}")

print("\n✅ Drive setup complete! Your research environment is structurally sound.")

In [ ]:
!pip install torch librosa transformers huggingface_hub soundfile

In [ ]:
!pip install fugashi unidic-lite sentencepiece

In [ ]:

import os
import warnings
import logging

import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import librosa
from google.colab import drive, userdata
from huggingface_hub import login
from transformers import (
    Wav2Vec2Model,
    Wav2Vec2FeatureExtractor,
    AutoModel,
    AutoTokenizer,
    AutoModelForSeq2SeqLM, # Added explicitly for translation
    pipeline,
)

# ── Silence non-fatal noise ────────────────────────────────────────────────────
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

torch.manual_seed(42)  # reproducibility


# ── Part 1: Initialisation ─────────────────────────────────────────────────────
def setup_environment() -> torch.device:
    drive.mount("/content/drive")
    project_root = "/content/drive/MyDrive/KOMOREBI"
    for folder in ("data", "checkpoints", "logs", "transcripts", "knowledge_graphs"):
        os.makedirs(os.path.join(project_root, folder), exist_ok=True)

    try:
        login(token=userdata.get("HF_TOKEN"))
        print("✅ KOMOREBI v2 Authentication Successful.")
    except Exception:
        print("⚠️  HF_TOKEN not found — some models may download slowly.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 System Online. Running on: {device}")
    return device


def _pipeline_device(device: torch.device) -> int:
    """Return the device index expected by HuggingFace pipeline()."""
    return 0 if device.type == "cuda" else -1


# ── Part 2: Fusion Model ───────────────────────────────────────────────────────
class KomorebiFusionModel(nn.Module):
    """
    Cross-attention fusion of Wav2Vec2 (acoustic) and BERT (linguistic) features
    for binary classification: Healthy (0) vs MCI (1).

    Forward args:
        audio_values  : (B, T_audio)   — raw 16kHz waveform values
        text_inputs   : dict            — tokenizer output (input_ids, attention_mask, …)
        audio_mask    : (B, T_audio) or None — 1 for valid frames, 0 for padding
    """

    ACOUSTIC_ID  = "reazon-research/japanese-wav2vec2-base"
    LINGUISTIC_ID = "cl-tohoku/bert-base-japanese-v3"
    HIDDEN_DIM    = 768
    NUM_CLASSES   = 2

    def __init__(self):
        super().__init__()
        self.acoustic_encoder  = Wav2Vec2Model.from_pretrained(self.ACOUSTIC_ID)
        self.linguistic_encoder = AutoModel.from_pretrained(self.LINGUISTIC_ID)

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=self.HIDDEN_DIM,
            num_heads=8,
            dropout=0.2,
            batch_first=True,
        )
        self.classifier = nn.Sequential(
            nn.Linear(self.HIDDEN_DIM, 256),
            nn.GELU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, self.NUM_CLASSES),
        )

    def forward(
        self,
        audio_values: torch.Tensor,
        text_inputs: dict,
        audio_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        # acoustic features: (B, T_a, 768)
        acoustic_out = self.acoustic_encoder(
            audio_values, attention_mask=audio_mask
        )
        a_feat = acoustic_out.last_hidden_state

        # linguistic features: (B, T_l, 768)
        l_feat = self.linguistic_encoder(**text_inputs).last_hidden_state

        # cross-attention: text queries attend to audio keys/values
        # key_padding_mask is (B, T_a) — True where frames should be IGNORED
        key_padding_mask = None
        if audio_mask is not None:
            # Wav2Vec2 output is subsampled; recompute mask length
            T_a = a_feat.size(1)
            # simple interpolation: mark padding positions
            ratio = audio_values.size(1) / T_a
            sub_mask = torch.nn.functional.avg_pool1d(
                audio_mask.float().unsqueeze(1), kernel_size=int(ratio), stride=int(ratio)
            ).squeeze(1)
            key_padding_mask = sub_mask < 0.5  # (B, T_a_sub)

        attn_output, _ = self.cross_attention(
            query=l_feat,
            key=a_feat,
            value=a_feat,
            key_padding_mask=key_padding_mask,
        )

        # CLS token representation → classifier
        cls_repr = attn_output[:, 0, :]
        return self.classifier(cls_repr)


# ── Part 3: Preprocessor ──────────────────────────────────────────────────────
class KomorebiPreprocessor:
    def __init__(self, device: torch.device):
        self.device = device
        self.audio_processor = Wav2Vec2FeatureExtractor.from_pretrained(
            KomorebiFusionModel.ACOUSTIC_ID
        )
        self.text_tokenizer = AutoTokenizer.from_pretrained(
            KomorebiFusionModel.LINGUISTIC_ID
        )

    def process_sample(
        self, audio_path: str, transcript: str
    ) -> tuple[torch.Tensor, dict]:
        speech, _ = librosa.load(audio_path, sr=16000, mono=True)

        audio_tensors = self.audio_processor(
            speech,
            return_tensors="pt",
            sampling_rate=16000,
        ).input_values.to(self.device)  # (1, T)

        text_tensors = self.text_tokenizer(
            transcript,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(self.device)

        return audio_tensors, text_tensors

    def process_batch(
        self, transcripts: list[str]
    ) -> dict:
        """Tokenise a list of transcripts into a batched dict on self.device."""
        return self.text_tokenizer(
            transcripts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(self.device)


# ── Part 4: ASR ───────────────────────────────────────────────────────────────
class KomorebiASR:
    """Transcribes Japanese speech to text via Whisper."""

    def __init__(self, device: torch.device):
        print("✅ Initialising Whisper ASR Module...")
        self.asr_pipeline = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-small",
            device=_pipeline_device(device),
            chunk_length_s=30,
        )

    def transcribe(self, audio_path: str) -> str:
        speech, sr = librosa.load(audio_path, sr=16000, mono=True)
        result = self.asr_pipeline(
            {"array": speech, "sampling_rate": sr},
            generate_kwargs={"language": "japanese"},
        )
        return result["text"]


# ── Part 5: English Subtitle Translator ───────────────────────────────────────
class KomorebiTranslator:
    """
    Translates Japanese text → English using Helsinki-NLP/opus-mt-ja-en.

    Runs entirely offline after the first download — no API key required.
    Used to provide English subtitles for all Japanese output so non-Japanese
    speakers can follow transcripts, RAG context, and training sample labels.

    Usage:
        translator = KomorebiTranslator(device)
        en = translator.translate("今日はいい天気ですね")
        # → "It's nice weather today."
    """

    MODEL_ID = "Helsinki-NLP/opus-mt-ja-en"

    def __init__(self, device: torch.device):
        print("✅ Initialising Japanese→English Translator (Helsinki-NLP)...")
        self.device = device
        # FIX: explicitly load tokenizer and model to bypass broken 'translation' pipeline in transformers v5
        self.tokenizer = AutoTokenizer.from_pretrained(self.MODEL_ID)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(self.MODEL_ID).to(self.device)

    def translate(self, japanese_text: str) -> str:
        """Translate a single Japanese string to English."""
        if not japanese_text or not japanese_text.strip():
            return ""
        inputs = self.tokenizer(
            japanese_text,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(self.device)

        with torch.inference_mode():
            outputs = self.model.generate(**inputs, max_new_tokens=512)

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def translate_batch(self, texts: list[str]) -> list[str]:
        """Translate a list of Japanese strings in one batched forward pass."""
        if not texts:
            return []
        inputs = self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(self.device)

        with torch.inference_mode():
            outputs = self.model.generate(**inputs, max_new_tokens=512)

        return [self.tokenizer.decode(out, skip_special_tokens=True) for out in outputs]


def _print_bilingual(label: str, japanese: str, english: str) -> None:
    """Helper: print a Japanese string with its English translation beneath it."""
    print(f"{label}")
    print(f"  🇯🇵  {japanese}")
    print(f"  🇬🇧  {english}")


# ── Part 6: Cultural Grounding RAG ────────────────────────────────────────────
class CulturalGroundingRAG:
    """
    Retrieval-Augmented Grounding using a small Showa-era (1950s) knowledge base.
    Embeddings are computed once at init; retrieval is a cosine-similarity lookup.

    Bug fix: added similarity threshold — if no entry is close enough,
    returns None instead of confidently returning the least-wrong entry.

    The knowledge base is stored in Japanese (required for embedding accuracy).
    Pass a KomorebiTranslator to retrieve_context() to also get an English subtitle.
    """

    SIMILARITY_THRESHOLD = 0.60

    def __init__(self, tokenizer, encoder: nn.Module, device: torch.device):
        print("✅ Cultural Knowledge Graph Module initialising...")
        self.tokenizer = tokenizer
        self.encoder   = encoder
        self.device    = device

        self.knowledge_base = [
            "昭和30年代の結婚式: 白無垢を着て、自宅で親族と行うのが一般的でした。",
            "昭和の遊び: メンコやコマ回し、お手玉など、外で遊ぶことが多かったです。",
            "昭和の食卓: ちゃぶ台を囲んで家族全員で食事をするのが当たり前でした。",
            "日本の天気挨拶: 挨拶代わりに天気の話題を出すのは、古くからの日本のコミュニケーションの基本です。",
        ]
        self.kb_embeddings = self._embed_texts(self.knowledge_base)

    def _embed_texts(self, texts: list[str]) -> torch.Tensor:
        inputs = self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(self.device)
        with torch.no_grad():
            hidden = self.encoder(**inputs).last_hidden_state
            # mean-pool over non-padding tokens
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            embeddings = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        return embeddings  # (N, 768)

    def retrieve_context(
        self,
        patient_utterance: str,
        translator: "KomorebiTranslator | None" = None,
    ) -> tuple[str | None, str | None]:
        """
        Returns (japanese_context, english_context).
        english_context is None if no translator is provided or no match found.
        """
        query_emb = self._embed_texts([patient_utterance])  # (1, 768)
        cos_sim   = nn.functional.cosine_similarity(query_emb, self.kb_embeddings)
        best_score, best_idx = cos_sim.max(dim=0)

        if best_score.item() < self.SIMILARITY_THRESHOLD:
            return None, None  # no confident match — avoids hallucination

        ja = self.knowledge_base[best_idx.item()]
        en = translator.translate(ja) if translator else None
        return ja, en


# ── Execution ─────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    device       = setup_environment()
    test_audio   = "/content/drive/MyDrive/KOMOREBI/data/sample_01.wav"

    print("⏳ Loading SOTA models (this may take a moment on Colab)…")
    model        = KomorebiFusionModel().to(device)
    preprocessor = KomorebiPreprocessor(device)
    asr          = KomorebiASR(device)
    translator   = KomorebiTranslator(device)
    rag          = CulturalGroundingRAG(
        preprocessor.text_tokenizer, model.linguistic_encoder, device
    )
    print("✅ KOMOREBI v2 Architecture fully compiled and ready.\n")

    # ── Generate dummy audio if needed ────────────────────────────────────────
    if not os.path.exists(test_audio):
        print(f"⚠️  {test_audio} not found — generating 2-second dummy file…")
        dummy = np.random.randn(32_000).astype(np.float32)
        sf.write(test_audio, dummy, 16000)

    # ── Step 1: ASR ───────────────────────────────────────────────────────────
    print("=" * 50)
    print("🚀 RUNNING FULL MULTIMODAL INFERENCE")
    print("=" * 50)

    print("🎙️  Step 1: Transcribing audio with Whisper ASR…")
    transcript    = asr.transcribe(test_audio)
    translation   = translator.translate(transcript)
    _print_bilingual("📝 Transcript:", transcript, translation)

    # ── Step 2: Preprocessing ─────────────────────────────────────────────────
    print("\n🎙️  Step 2: Preprocessing audio + text tensors…")
    audio_tensor, text_tensor = preprocessor.process_sample(test_audio, transcript)

    # ── Step 3: Fusion inference ───────────────────────────────────────────────
    print("🧠 Step 3: Cross-attention fusion…")
    model.eval()
    with torch.inference_mode():
        logits = model(audio_tensor, text_tensor)
        probs  = nn.functional.softmax(logits, dim=-1)

    print("\n✅ MULTIMODAL DIAGNOSTIC RESULTS")
    print(f"   Raw logits      : {logits.tolist()[0]}")
    print(f"   Probabilities   : Healthy={probs[0,0]:.3f}  MCI={probs[0,1]:.3f}")

    # ── Cultural RAG ──────────────────────────────────────────────────────────
    print("\n" + "=" * 50)
    print("⛩️  SHOWA-ERA CULTURAL GROUNDING (RAG)")
    print("=" * 50)
    context_ja, context_en = rag.retrieve_context(transcript, translator)
    _print_bilingual("🗣️  Patient said:", transcript, translation)
    if context_ja:
        _print_bilingual("📚 RAG context retrieved:", context_ja, context_en or "—")
    else:
        print("📚 No confident cultural context found (similarity below threshold).")

    # ── Clinical fine-tuning simulator ────────────────────────────────────────
    print("\n" + "=" * 50)
    print("🏥 CLINICAL FINE-TUNING SIMULATOR")
    print("=" * 50)

    # 4 synthetic patients: labels 0=Healthy, 1=MCI
    dummy_labels = torch.tensor([0, 1, 0, 1], device=device)
    dummy_audio  = torch.randn(4, 32_000, device=device)

    transcripts = [
        "今日はいい天気ですね",
        "えっと、あれが出ない",
        "ご飯を食べました",
        "昨日、あの、何だっけ",
    ]
    labels_str = ["Healthy", "MCI", "Healthy", "MCI"]

    print("📋 Training samples with English subtitles:")
    translations = translator.translate_batch(transcripts)
    for ja, en, lbl in zip(transcripts, translations, labels_str):
        print(f"  [{lbl}]")
        print(f"    🇯🇵  {ja}")
        print(f"    🇬🇧  {en}")
    print()
    batch_text = preprocessor.process_batch(transcripts)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(1, 4):
        optimizer.zero_grad()
        logits = model(dummy_audio, batch_text)
        loss   = criterion(logits, dummy_labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        print(f"Epoch {epoch}/3 | Loss: {loss.item():.4f}")

    print("\n✅ Project KOMOREBI — complete.")